VECTORIZATION CON SCIKIT-LEARN

Arriva ora il momnento di dare in pasto i dati ad un algoritmo
La vectorizatin con scikit-learn è il passaggio con cui trasformi il testo in numeri, così che un algoritmo di Machine Learning possa elaborarlo
testo -> vectorizer -> matrice numerica -> modello ml

Bag of Words e TF-IDF visti in precedenza sono due modi diversi per realizzare la vectorizzazione, scikit-learn da gli strumenti Python per applicare uno di questi due moetodi non in modo manuale (come fatto nelle lezioni precedenti) ma con strumenti specifici: CountVectorizer() per il Bag of Words, e TfidVectorizer() per il TF_IDF

Come sappiamo il computer non può prendere in pasto direttamente le stringhe a una Logistic Regression o a una SVM. Devo prima trasformarle in vettori numerici

Il metodo più semplice è CountVectorizer passandogli le frasi da vettorizzare
Si ottiene un Bag of Words
Il metodo importante di CountVectorizer è .fit_transform()
x=vectorizer.fit_transform(documenti)
fai due cose:
1) fit: legge tutti i documenti e costruisce il vocabolario, cioè impare COME rappresentare il testo
2) transform: prende ogni documento, lo converte usando il vocabolario del punto 1, produce il vettore numerico, cioè APPLICO la trasformazione imparata al punto 1
Pertanto, se dovessi ricevere un nuovo testo, non devo rifare fit_transform, perchè cambieresti il vocabolario, devi fare solo .transform(nuovo_testo)

In questa istruzione: x=vectorizer.fit_transform(documenti)
x non è un array NumPy ma una SPARSE MATRIX (print(type(x)))
Immagina un vocabolario di 50.000 parole, ed una mail con solo 100 parole il vettore potrebbe essere [0,0,0,1,0,0,0,2,0,0,0,...] quindi con migliaia di zeri.
Memorizzare tutti questi zeri sarebbe uno spreco enorme, scikit-learn usa quind una matrice sparsa (sparse matrix) che conserva principalmente i valori diversi da zero.
Puoi poi usare x (sparse matrix) in un classificatore (es. logisticregression)
Quindi una pipeline NLP classica può essere:
email -> countvectorizer (.fit_transform o solo transform)  -> sparse matrix -> logistic regression -> ordine/reclamo/offerta

CountVectorizer di scikit-learn è quindi una libreria standard per il machine learning classico in Python. E' lo strumento per automatizzare la creazione del vocabolario e la generazione della matrice dei conteggi. Agisce internamente la tokenizzazione e la rimozione della punteggiatura, producendo matrice sparse che ottimizzano l'occupazione di memoria (pulisce, tokenizza, e costriusce il vocabolario). 

Parametri Fondamentali
- Metodo fit_transform() esegue simultaneamente l'apprendimento del vocabolario (fit) e la trasformazione del testo in vettori (tranform)
- Attributo vocabulary_: un dizionario che mappa ogni parola del corpus a un indice intero specifico nella matrice risultante
- Parametro 'stop_words': permette di integrare liste di parole da ignorare direttamente nella fase di vettorizzazione
- Parametro 'ngram_range' possiamo dire al vettorizzatore di includere non solo singole parole, ma anche bigrammi e trigrammi nel vocabolario
- parametri min_df e max_df per escludere termini troppo rari o troppo frequenti in base alla percentuale di documenti in cui compaiono
- binary=True: l'oggetto ignora i conteggi multipli, registrando slo presenza (1) o assenza (0) di una parola.

Ogni riga i della sparse matrix rappresenta un documento codificato come un vettore di frequenze.
Ogni colonna della sparse matrix rappresenta una parola specifica del vocabolario

Ma Countvectorizer di scikit-learn non considera quanto sia veramente informativa una parola (limite di Bag of Words), ed è qui che entra in gioco TfidfVectorizer (TF-IDF)
TF-IDF cerca di dare peso basso a parole molto comuni nel corpus, peso alto a parole particolarmente informative in quel documento
Esempio in 10.000 emal commerciali la parola 'cliente' potrebbe comparire quasi ovunque, mentre la parola 'danneggiato' potrebbe comparire quasi esclusivamente nei reclami. Quindi TF-IDF considera 'cliente' meno discriminante e 'danneggiato' più discriminante 

Con TfidfVectorizer si applica la pesatura delle parole, riducendo le dimensioni dello spazio vettoriale per rendere i modelli pi+ leggere e informativi
L'uso corretto di 'max_features' è poi fondamentale per limitare la dimensionalità, permettendoci di focalizzare l'attenzione del modello solo su termini significativi.

- max_feature: limita il vocabolario ai primi 'k' termini ordinati per frequenza nel corpus
- tfidfvectorizer assicura che parole rare ma informative ricevano pesi maggiori rispetto a termini presenti in tutti i documenti
- normalizzazione L2: di default scikit-learn riscala i vettori TF-IDF affinchè abbiabo norma unitaria

Mantenere troppe feature (es. 50.000 parole per 1.000 documenti) porta il modello a memorizzare il rumore anzichè apprendere i pattern semantici generali. Riducendo il numero di feature velocizza drasticamente l'addestramento di classificatori come SVM o Logistic Regression. In molti compiti NLP classico, valore di max_features compresi tra 1.000 e 5.000 sono sufficienti per coprire la maggior parte del segnale informativo.


Per una classificazione testuale classica è preferibile usare TF-IDF con Logistic Regresion oppure TF-IDF con SVM piuttosto che il pure conteggio di Bag of Words.

E' possibile includere anche il concetto di n-grammi con CountVetorizer in modo da unire le parole di 'richiesta offerta'
Con CountVectorizer puoi anche aggiungere stopword, ma per l'italiano normalmente devi fornire una lista personalizzata.
Puoi impostare una soglia minima, considera solo i termini con una frequenza minima es min_df=2 (considera sola i termini presenti in almeno 2 documenti)
Puoi impostare una soglia massima, max_df=0.9, ignora i termini presenti in oltre il 90% dei documenti 

Riassumendo:
La vectorization in scikit-learn trasforma un insieme di documenti testuali in una matrice numerica di feature. CountVectorizer usa le frequenze dei termini, mentre TfidfVectorizer ne pesa anche l'importanza rispetto all'intero corpus
Sul training fai fit_transform(). Sui nuovi testi fai soltanto transform().

Abbiamo quindi addestrato il nostro vettorizzatore, ma cosa succede quando spegni il computer?

Salvataggio e Inferenza
Garantire la coerenza del vocabolario in produzione
Una volta addestrato un modello NLP, dobbiamo esser in grado di processare nuovi dati (test set o input dell'utente) nello stesso specifico modo. Se il vocabolario del vettorizzatore cambi, gli indici dei vettori non corrisponderanno più a quelli attesi dal modello. Il vocabolario deve rimanere identico tra la fase di training e la fase di inferenza.
L'oggetto vettorizzatore va salvato su disco pe essere poi caricato su disco durante la fase di inferenza e garantire che la mappatura word-to-id rimanga costante.

Persistenza degli Oggetti
Mai rieseguire il metodo .fit() sui dati di test, si deve utilizzare solo transform() questo per non inquinare il modello. L'apprendimento del vocabolario deve avvenire esclusivamente sui dati di training per non inquinare il modello.
Serializzare con Joblib per salvare efficientemente oggetti Python che contengono grandi array numpy.
La pipeline di inferenza carica il vettorizzatore salvato 'V' e applica la trasformazione ai nuovi dati 'X_new'

Ogni volta che si riaddestra il modello su nuovi dati (.fit o .fitransform), è necessario salvare una nuova versione del vettorizzatore, poichè il vocabolario potrebbe essere cambiato.
L'uso della classe 'Pipeline' di scikit-leasrn permette di impacchettare vettorizzatore e modello in un unico oggetto serializzabile.
Le parole nuove incontrate in produzione verranno ignorate dal vettorizzatore poichè non mappate nel vocabolario originale caricato.

e se lo spazio su disco scarseggia?

Joblib ha un'ultima carta da giocare, joblib è ottimizzato per la persistenza di oggetti che contengono strutture dati sparse e pesanti tipiche di scikit-learn
L'uso della compressione permette di ridurre l'ingombro dei file 'pkl' senza perdere informazioni critiche sul vocabolario
In contesti di Deep Learning, avere a disposizione una compressione ottimizzata può essere ciò che fa la differenza, assicurati sempre che l'ambiente di caricamente sia compatibile con quello di salvataggio per evitare errori di versione


In [ ]:
"""
PIPELINE DI VECTORIZATION CON SCIKIT-LEARN
------------------------------------------
Questo script dimostra il workflow per trasformare testo non strutturato
in rappresentazioni numeriche (vettori) utilizzabili dai modelli di Machine Learning.

Concetti chiave:
1. Tokenizzazione: Divisione del testo in singole parole o gruppi di parole (n-grammi).
2. Vocabolario: Mappatura univoca tra parole e indici numerici.
3. Vettorizzazione: Trasformazione di documenti in righe di una matrice sparsa.
"""

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import joblib  # Standard per la serializzazione efficiente di oggetti scikit-learn

# ==========================================
# 1. DEFINIZIONE DEL CORPUS (Dataset di Testo)
# ==========================================
# Il "Corpus" è la collezione completa dei documenti che useremo per "istruire" 
# il nostro vettorizzatore. Da qui verrà estratto il vocabolario globale.
corpus_train = [
    "L'intelligenza artificiale sta cambiando il mondo.",
    "Il deep learning è una sottocategoria dell'intelligenza artificiale.",
    "Le reti neurali sono alla base dei modelli di deep learning.",
    "Python è il linguaggio principale per la data science."
]

# =============================================================
# 2. COUNT-VECTORIZATION (Bag of Words - BoW)
# =============================================================
# Classe: CountVectorizer
# Scopo: Conta semplicemente quante volte ogni parola appare in ogni documento.

# Parametro ngram_range=(1, 2): 
# - Include unigrammi ("intelligenza") 
# - Include bigrammi ("intelligenza artificiale")
# Utile per catturare il contesto locale dove l'ordine delle parole conta.
count_vec = CountVectorizer(ngram_range=(1, 2))

# METODO fit_transform:
# 1. .fit(): Analizza il corpus_train e costruisce il vocabolario (ID <-> Parola).
# 2. .transform(): Converte ogni frase in un vettore di conteggi basato sul vocabolario creato.
# Risultato: Una matrice SciPy CSR (Compressed Sparse Row) per risparmiare memoria (piena di zeri).
X_counts = count_vec.fit_transform(corpus_train)

print(f"--- SEZIONE 1: CountVectorizer ---")
print(f"Dimensioni matrice BoW (Documenti, Termini): {X_counts.shape}")
# get_feature_names_out(): Estrae la lista ordinata delle parole che compongono le colonne della matrice.
print(f"Top 5 termini nel vocabolario: {count_vec.get_feature_names_out()[:5]}")


# =============================================================
# 3. TF-IDF E OTTIMIZZAZIONE DELLE FEATURE
# =============================================================
# Classe: TfidfVectorizer
# Scopo: Pesa le parole non solo in base alla frequenza (TF), ma anche alla loro 
# rarità nel corpus (IDF - Inverse Document Frequency). Parole comuni come "il" 
# avranno peso basso; parole specifiche come "neurali" avranno peso alto.

# Parametro max_features=10:
# Seleziona solo i 10 termini con il punteggio TF-IDF più alto in tutto il dataset.
# Strategia cruciale per ridurre il "rumore" e la complessità computazionale.
tfidf_vec = TfidfVectorizer(max_features=10, stop_words=None)

# Eseguiamo la trasformazione statistica
X_tfidf = tfidf_vec.fit_transform(corpus_train)

# La matrice risultante ha righe normalizzate (L2 normalization):
# Ogni riga è un vettore la cui somma dei quadrati degli elementi è 1.
print(f"\n--- SEZIONE 2: TF-IDF ---")
print(f"Feature più rilevanti selezionate: {tfidf_vec.get_feature_names_out()}")


# =============================================================
# 4. PERSISTENZA E SERIALIZZAZIONE (Model Saving)
# =============================================================
# Perché salvare? In produzione non avremo il corpus originale. 
# Dobbiamo usare LO STESSO vocabolario e pesi IDF per trasformare i nuovi dati.
model_filename = "tfidf_vectorizer_v1.joblib"

# joblib.dump: Salva l'oggetto Python (lo stato interno del vettorizzatore) in un file binario.
joblib.dump(tfidf_vec, model_filename)
print(f"\n[INFO] Vettorizzatore 'fit' salvato in: {model_filename}")


# =============================================================
# 5. FASE DI INFERENZA (Utilizzo su nuovi dati)
# =============================================================
# Simuliamo l'applicazione del modello su un server o in una nuova sessione.

# 5.1 Caricamento dell'oggetto precedentemente addestrato
loaded_vec = joblib.load(model_filename)

# 5.2 Nuovo input (mai visto durante l'addestramento)
new_input = ["L'intelligenza artificiale usa le reti neurali."]

# 5.3 TRASFORMAZIONE (Cruciale: SENZA .fit())
# Usiamo SOLO .transform() perché:
# - Se usassimo .fit(), creeremmo un nuovo vocabolario ignorando quello di training.
# - Se l'input contiene parole nuove (es. "usa"), esse verranno ignorate per mantenere
#   la compatibilità con le dimensioni attese dal modello di ML a valle.
X_new = loaded_vec.transform(new_input)

print("\n--- SEZIONE 3: Inferenza ---")
print("Vettore numerico generato per il nuovo input (rappresentazione sparsa):")
print(X_new)
print("\nSpiegazione: Gli indici sopra corrispondono ai termini nel vocabolario caricato.")